In [0]:
import subprocess
subprocess.run(["pip", "install", "openpyxl", "--quiet"], check=True)

import io
import unicodedata
import re
import logging
from difflib import SequenceMatcher

import boto3
import pandas as pd
import openpyxl
from pyspark.sql.functions import current_timestamp, lit

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ── Pipeline config
S3_BUCKET     = "ss2-ingestion-ine-datasets-raw"
S3_PREFIX     = "ine_defunciones/"
TARGET_SCHEMA = "sandbox_bronze_ss2"
SOURCE_SYSTEM = "INE_DEFUNCIONES"
WRITE_MODE    = "overwrite"   

AWS_ACCESS_KEY    = dbutils.secrets.get(scope="ss2-bronze-layer", key="aws_access_key_id")
AWS_SECRET_KEY    = dbutils.secrets.get(scope="ss2-bronze-layer", key="aws_secret_access_key")
AWS_SESSION_TOKEN = None

CONTENTS_TAB        = "contenido"
HEADER_SCAN_ROWS    = 15    # max rows to scan looking for the real data header
FUZZY_MATCH_CUTOFF  = 0.60  # minimum similarity ratio to accept a sheet/content match
MIN_HEADER_COLS     = 2     # a valid header row must have at least this many non-null cells

# Metadata-like patterns found in INE header blocks — rows matching these are skipped
# when searching for the real column header row.
METADATA_PATTERNS = re.compile(
    r'^(instituto|direcci[oó]n|departamento de estad|cuadro\s*\d|año\s*\d{4}|regresar)',
    re.IGNORECASE
)

# Sheets to extract from contenido tab → target Delta table.
TARGET_SHEETS = {
    "Defunciones por sexo, según edad y causas de muerte":
        f"{TARGET_SCHEMA}.ine_defunciones_sexo_edad_causas_muerte",
    "Defunciones por sexo, según departamento de residencia del difunto(a) y causas de muerte":
        f"{TARGET_SCHEMA}.ine_defunciones_sexo_depto_residencia_causas_muerte",
    "Defunciones neonatales por sexo, según edad y causas de muerte":
        f"{TARGET_SCHEMA}.ine_defunciones_neonatales_sexo_edad_causas_muerte",
    "Defunciones post-neonatales por sexo, según edad y causas de muerte":
        f"{TARGET_SCHEMA}.ine_defunciones_postneonatales_sexo_edad_causas_muerte",
    "Defunciones por causas externas y sexo, según departamento de ocurrencia":
        f"{TARGET_SCHEMA}.ine_defunciones_causas_externas_sexo_depto_ocurrencia",
}


# ── Helpers 

def list_xlsx_keys(bucket, prefix):
    """
    Lists xlsx files under s3://bucket/prefix using dbutils.fs.ls (recursive).
    Returns: {year_str: s3_key, ...}
    """
    root       = f"s3://{bucket}/{prefix}"
    year_files = {}

    def _walk(path):
        for f in dbutils.fs.ls(path):
            if f.isDir():
                _walk(f.path)
            elif f.name.lower().endswith(".xlsx"):
                parts     = f.path.replace(f"s3://{bucket}/", "").split("/")
                year_part = next((p for p in parts if p.startswith("year=")), None)
                if year_part is None:
                    logger.warning(f"  Skipping file with no year= partition: {f.path}")
                    return
                year = year_part.replace("year=", "")
                if year not in year_files:
                    key = f.path.replace(f"s3://{bucket}/", "")
                    year_files[year] = key
                    logger.info(f"  Found xlsx for year {year}: {f.path}")

    _walk(root)
    return year_files


def download_xlsx_bytes(bucket, key):
    """
    Downloads an xlsx from S3 via boto3 using explicit AWS credentials
    stored in Databricks Secrets. Works on Spark Connect / serverless
    clusters where sparkContext and JVM access are unavailable.
    """
    kwargs = dict(
        aws_access_key_id     = AWS_ACCESS_KEY,
        aws_secret_access_key = AWS_SECRET_KEY,
    )
    if AWS_SESSION_TOKEN:
        kwargs["aws_session_token"] = AWS_SESSION_TOKEN

    client   = boto3.client("s3", **kwargs)
    response = client.get_object(Bucket=bucket, Key=key)
    return response["Body"].read()


def clean_name(name):
    name = str(name).lower()
    name = name.replace(".xlsx", "").replace(".xls", "").replace(".csv", "")
    name = name.replace("ñ", "ni")
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-z0-9_]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def dedupe_columns(cols):
    seen   = {}
    result = []
    for col in cols:
        count     = seen.get(col, 0)
        seen[col] = count + 1
        result.append(col if count == 0 else f"{col}_{count + 1}")
    return result


def normalize_columns(df):
    df.columns = dedupe_columns([clean_name(c) for c in df.columns])
    df = df.loc[:, ~df.columns.str.match(r'^unnamed_\d+$')]
    return df


def fix_mixed_type_columns(df):
    """Cast predominantly-numeric object columns to float; rest to str."""
    skip = {"anio", "source_file", "source_system"}
    for col in df.select_dtypes(include="object").columns:
        if col in skip:
            continue
        coerced  = pd.to_numeric(df[col], errors="coerce")
        non_null = df[col].notna().sum()
        if non_null > 0 and coerced.notna().sum() / non_null >= 0.9:
            df[col] = coerced
        else:
            df[col] = df[col].where(df[col].isna(), df[col].astype(str))
    return df


def similarity(a, b):
    """Returns a 0-1 similarity ratio between two strings (case-insensitive, stripped)."""
    return SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio()


# ── Step 1: Read contenido tab ────────────────────────────────────────────────

def get_contenido_names(workbook):
    """
    Reads the 'contenido' tab and returns all non-empty text values from column B (index 1).
    These are the canonical sheet descriptions used to locate the actual data tabs.
    Returns: list of str
    """
    ws_name = next(
        (s for s in workbook.sheetnames if s.strip().lower() == CONTENTS_TAB),
        None
    )
    if ws_name is None:
        logger.warning("  'contenido' tab not found in workbook.")
        return []

    ws    = workbook[ws_name]
    names = []
    for row in ws.iter_rows(min_row=2, values_only=True):
        cell = row[1] if len(row) > 1 else None
        if cell and str(cell).strip():
            names.append(str(cell).strip())

    logger.info(f"  'contenido' lists {len(names)} entries: {names}")
    return names


# ── Step 2 & 3: Resolve sheet name using cascade strategy ────────────────────

def resolve_sheet_name(workbook, target_name, contenido_names):
    """
    Finds the actual workbook sheet that corresponds to `target_name` using a
    3-level cascade:

    Level 1 — Exact match between target_name and a workbook sheet name
              (case-insensitive, stripped).

    Level 2 — Position-based match via contenido tab.
              The contenido tab lists sheets in the same order as the workbook.
              We find the position of target_name in contenido (fuzzy, >=0.85)
              and return the workbook sheet at that same position (skipping the
              contenido sheet itself). This is the most reliable method since
              INE uses abbreviated tab names that don't match the full names.

    Level 3 — Cell content scan: reads the first HEADER_SCAN_ROWS rows of each
              sheet looking for a title cell similar to target_name. Fallback
              for files where contenido order doesn't align.

    Returns: matched sheet name (str) or None.
    """
    sheets = workbook.sheetnames
    # Non-contenido sheets in workbook order (these align with contenido rows)
    data_sheets = [s for s in sheets if s.strip().lower() != CONTENTS_TAB.lower()]

    # ── Level 1: exact match on sheet tab name ────────────────────────────────
    target_norm = target_name.strip().lower()
    for s in sheets:
        if s.strip().lower() == target_norm:
            logger.info(f"    [L1-exact] '{target_name}' → sheet '{s}'")
            return s

    # ── Level 2: position via contenido tab ───────────────────────────────────
    # Find which entry in contenido best matches target_name
    if contenido_names:
        best_idx, best_ratio = -1, 0.0
        for idx, name in enumerate(contenido_names):
            ratio = similarity(target_name, name)
            if ratio > best_ratio:
                best_ratio, best_idx = ratio, idx

        if best_ratio >= 0.85 and best_idx < len(data_sheets):
            matched = data_sheets[best_idx]
            logger.info(
                f"    [L2-pos  ] '{target_name}' → contenido[{best_idx}]='{contenido_names[best_idx]}' "
                f"(ratio={best_ratio:.2f}) → sheet '{matched}'"
            )
            return matched

    # ── Level 3: scan cell content inside each sheet ──────────────────────────
    logger.info(f"    [L3-scan ] No position match for '{target_name}'; scanning cell content...")
    best_sheet, best_ratio = None, 0.0
    for s in data_sheets:
        ws = workbook[s]
        for row in ws.iter_rows(max_row=HEADER_SCAN_ROWS, values_only=True):
            for cell in row:
                if cell is None:
                    continue
                cell_text = str(cell).strip()
                if len(cell_text) < 10:
                    continue
                ratio = similarity(target_name, cell_text)
                if ratio > best_ratio:
                    best_ratio, best_sheet = ratio, s
        if best_ratio >= FUZZY_MATCH_CUTOFF:
            break

    if best_ratio >= FUZZY_MATCH_CUTOFF:
        logger.info(f"    [L3-scan ] '{target_name}' → sheet '{best_sheet}' (ratio={best_ratio:.2f})")
        return best_sheet

    logger.warning(f"    [MISS    ] No sheet found for '{target_name}' (best ratio={best_ratio:.2f})")
    return None


# ── Step 4: Detect the real header row inside a sheet ────────────────────────

def find_header_row(data, max_scan=HEADER_SCAN_ROWS):
    """
    Scans the first `max_scan` rows of a sheet (as a list-of-tuples) and returns
    the index of the first row that looks like a real column header:

    - Has at least MIN_HEADER_COLS non-null cells
    - The first non-null cell does NOT match METADATA_PATTERNS
    - All non-null values in the row are strings (headers are never pure numbers)
    - Spans multiple columns (not a single merged title cell)

    Handles two known INE layouts:
      Variant A: several metadata rows → header row with dark background
      Variant B: first row is a merged title → second row is the real header

    Returns: (header_row_index, data_start_index) or (None, None) if not found.
    """
    for i, row in enumerate(data[:max_scan]):
        non_null = [c for c in row if c is not None and str(c).strip() != ""]

        # Must have enough columns to be a header
        if len(non_null) < MIN_HEADER_COLS:
            continue

        # All values must be text (numeric first cell → likely a data row, not header)
        if any(_is_numeric(c) for c in non_null):
            continue

        # First cell must not look like INE boilerplate metadata
        first_text = str(non_null[0]).strip()
        if METADATA_PATTERNS.match(first_text):
            continue

        # Single-cell rows that span the full width are merged titles (Variant B row 0)
        # They pass the text check but have only 1 distinct non-null value repeated
        unique_vals = set(str(c).strip() for c in non_null)
        if len(unique_vals) == 1 and len(non_null) > 1:
            continue  # merged title row — skip and keep scanning

        logger.info(f"    Header detected at row index {i}: {[str(c) for c in non_null[:6]]}")
        return i, i + 1   # header at i, data starts at i+1

    return None, None


def _is_numeric(value):
    """Returns True if value can be interpreted as a number."""
    try:
        float(str(value).replace(",", "").replace(" ", ""))
        return True
    except (ValueError, TypeError):
        return False


# ── Main sheet reader ─────────────────────────────────────────────────────────

def read_target_sheet(workbook, target_name, contenido_names, anio, source_file, source_system):
    """
    Full cascade for one target sheet:
      1. Resolve which workbook sheet to open (levels 1-3)
      2. Detect the real header row intelligently (handles both INE layouts)
      3. Return a clean DataFrame with anio + source_file columns, or None.
    """
    sheet_name = resolve_sheet_name(workbook, target_name, contenido_names)
    if sheet_name is None:
        return None

    ws   = workbook[sheet_name]
    data = list(ws.values)

    if len(data) < 2:
        logger.warning(f"    Sheet '{sheet_name}' has fewer than 2 rows; skipping.")
        return None

    header_idx, data_idx = find_header_row(data)
    if header_idx is None:
        logger.warning(f"    Could not detect header row in '{sheet_name}'; skipping.")
        return None

    raw_headers = data[header_idx]
    rows        = data[data_idx:]

    # Find the last real header column — Excel often includes extra empty/formatted
    # cells beyond the actual table. We trim at the last non-empty header cell.
    last_real_col = 0
    for i, h in enumerate(raw_headers):
        if h is not None and str(h).strip() != "":
            last_real_col = i
    raw_headers = raw_headers[:last_real_col + 1]

    # Build column names only up to last_real_col; no col_X placeholders for
    # trailing empties since we already trimmed them above.
    headers = [
        str(h).strip() if h is not None and str(h).strip() != "" else f"col_{i}"
        for i, h in enumerate(raw_headers)
    ]

    # Truncate each data row to match the header length
    rows = [row[:last_real_col + 1] for row in rows]

    df = pd.DataFrame(list(rows), columns=headers)
    df = normalize_columns(df)

    # Drop any remaining col_N columns (empty headers in the middle of the table)
    df = df.loc[:, ~df.columns.str.match(r"^col_\d+$")]

    # Drop fully-null columns (Excel ghost columns with no data at all)
    df.dropna(axis=1, how="all", inplace=True)

    df.dropna(how="all", inplace=True)

    # Drop rows where all data columns (excluding metadata) are null
    data_cols = [c for c in df.columns if c not in {"anio", "source_file"}]
    df.dropna(subset=data_cols, how="all", inplace=True)

    df["anio"]          = int(anio)
    df["source_file"]   = source_file
    df["source_system"] = source_system

    logger.info(
        f"    Sheet '{sheet_name}': header@row{header_idx}, "
        f"{len(df):,} rows, {len(df.columns)} cols → {list(df.columns)}"
    )
    return df


# ── Core ingestion ────────────────────────────────────────────────────────────

def ingest_sheet(target_name, table_name, year_files, write_mode, source_system):
    """
    Reads `target_name` from every year xlsx and writes the combined
    result to a single Delta table.
    """
    logger.info("=" * 70)
    logger.info(f"Target : {target_name}")
    logger.info(f"Table  : {table_name}  [mode={write_mode}, years={sorted(year_files)}]")

    frames = []
    for anio, key in sorted(year_files.items()):
        source_file = f"s3://{S3_BUCKET}/{key}"
        try:
            logger.info(f"  Year {anio} ← {source_file}")
            xlsx_bytes = download_xlsx_bytes(S3_BUCKET, key)
            wb = openpyxl.load_workbook(
                io.BytesIO(xlsx_bytes), read_only=True, data_only=True
            )

            # Step 1: get canonical names from contenido tab
            contenido_names = get_contenido_names(wb)

            # Steps 2-4: resolve + read the sheet
            df = read_target_sheet(wb, target_name, contenido_names, anio, source_file, source_system)
            wb.close()

            if df is not None and not df.empty:
                frames.append(df)

        except Exception as exc:
            logger.warning(f"  Skipped year {anio} ({key}): {exc}")

    if not frames:
        logger.warning(f"  No valid data collected for '{target_name}'; skipping write.")
        return

    combined         = pd.concat(frames, ignore_index=True)
    combined.columns = dedupe_columns(list(combined.columns))
    combined         = fix_mixed_type_columns(combined)
    combined["anio"] = combined["anio"].astype(int)

    logger.info(
        f"  Total: {len(combined):,} rows across "
        f"{combined['anio'].nunique()} years → {table_name}"
    )

    sdf = spark.createDataFrame(combined)
    sdf = sdf.withColumn("ingestion_timestamp", current_timestamp())

    (sdf.write
        .format("delta")
        .mode(write_mode)
        .option("mergeSchema", "true")
        .saveAsTable(table_name))

    logger.info(f"  Wrote {len(combined):,} rows → {table_name}")


# ── Entry point ───────────────────────────────────────────────────────────────
try:
    year_files = list_xlsx_keys(S3_BUCKET, S3_PREFIX)

    if not year_files:
        raise Exception(f"No xlsx files found under s3://{S3_BUCKET}/{S3_PREFIX}")

    logger.info(f"Found {len(year_files)} year partitions: {sorted(year_files.keys())}")

    for target_name, table_name in TARGET_SHEETS.items():
        ingest_sheet(
            target_name   = target_name,
            table_name    = table_name,
            year_files    = year_files,
            write_mode    = WRITE_MODE,
            source_system = SOURCE_SYSTEM,
        )

    logger.info("Pipeline completed successfully.")
except Exception as e:
    logger.error(f"Pipeline failed: {e}")
    raise
